In [7]:
import os
import subprocess
import time
import requests

In [8]:
# Configuration
NGROK_TOKEN = "3LbdNxIv7nXzBYkFHoLxeKqVsXj_4j3hnBwn61fxFDBydr1W3"
PORT = "9931"

# Install llama.cpp
subprocess.run(
    "curl -LsSf https://llama.app/install.sh | sh",
    shell=True, check=True
)

# Install ngrok
subprocess.run(
    "curl -fsSL https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar -xz",
    shell=True, check=True
)
# Authenticate ngrok
subprocess.run(
    ["./ngrok", "config", "add-authtoken", NGROK_TOKEN],
    check=True
)


Version: b12110
Probing CUDA...
Found: 75
Installation completed successfully



To make llama available in future sessions, add ~/.local/bin to your PATH by running:

  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bash_profile

Then open a new terminal and run:

  llama serve

To start it now without modifying your PATH, run:

  ~/.llama-app/llama serve

Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml


CompletedProcess(args=['./ngrok', 'config', 'add-authtoken', '3LbdNxIv7nXzBYkFHoLxeKqVsXj_4j3hnBwn61fxFDBydr1W3'], returncode=0)

In [9]:
import sys
import subprocess
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip",
    "install", "-q", "-U", "huggingface_hub"
], check=True)

from huggingface_hub import hf_hub_download

models = {
    "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF":
        "Huihui-Qwen3.8-27B-abliterated-Q6_K.gguf",
    "mradermacher/Huihui-Qwen3.5-9B-abliterated-GGUF":
        "Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf"
}

Path("/content/models").mkdir(exist_ok=True)

for repo, filename in models.items():
    print(f"Downloading: {filename}", flush=True)
    try:
        path = hf_hub_download(
            repo_id=repo,
            filename=filename,
            local_dir="/content/models"
        )
        print(f"Downloaded: {path}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

print("All requested downloads completed!")

Downloading: Huihui-Qwen3.8-27B-abliterated-Q6_K.gguf
Downloaded: /content/models/Huihui-Qwen3.8-27B-abliterated-Q6_K.gguf
Downloading: Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf


Huihui-Qwen3.5-9B-abliterated.Q4_K_M.ggu(…): reconstructing file:   0%|          |  0.00B / 5.63GB            

Huihui-Qwen3.5-9B-abliterated.Q4_K_M.ggu(…): downloading bytes:           |  0.00B            

Downloaded: /content/models/Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf
All requested downloads completed!


In [10]:
import os
import subprocess
import time
import requests

PORT = "9931"
MODELS_DIR = "/content/models"

# Stop previous llama server
if "llama" in globals() and llama.poll() is None:
    llama.terminate()
    llama.wait()

# Start multi-model router
llama = subprocess.Popen(
    [
        os.path.expanduser("~/.llama-app/llama"),
        "serve",
        "--models-dir", MODELS_DIR,
        "--host", "127.0.0.1",
        "--port", PORT
    ],
    stdout=open("/tmp/llama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Verify both models are registered
response = requests.get(
    f"http://127.0.0.1:{PORT}/v1/models"
)
print(response.json())

{'data': [{'id': 'Huihui-Qwen3.5-9B-abliterated.Q4_K_M', 'aliases': [], 'tags': [], 'object': 'model', 'owned_by': 'llamacpp', 'created': 1793907709, 'status': {'value': 'unloaded', 'args': ['/root/.llama-app/llama', 'serve', '--host', '127.0.0.1', '--log-colors', 'off', '--port', '0', '--alias', 'Huihui-Qwen3.5-9B-abliterated.Q4_K_M', '--model', '/content/models/Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf'], 'preset': '[Huihui-Qwen3.5-9B-abliterated.Q4_K_M]\nlog-colors = off\nmodel = /content/models/Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf\n\n'}, 'architecture': {'input_modalities': ['text'], 'output_modalities': ['text']}, 'source': 'models_dir', 'can_remove': False, 'context_length': 262144}, {'id': 'Huihui-Qwen3.8-27B-abliterated-Q6_K', 'aliases': [], 'tags': [], 'object': 'model', 'owned_by': 'llamacpp', 'created': 1793907709, 'status': {'value': 'unloaded', 'args': ['/root/.llama-app/llama', 'serve', '--host', '127.0.0.1', '--log-colors', 'off', '--port', '0', '--alias', 'Huihui-Q

In [11]:
import subprocess
import time
import requests
import os

# Ensure ngrok binary is executable
subprocess.run(["chmod", "+x", "./ngrok"], check=True)

# Start ngrok
ngrok = subprocess.Popen(
    ["./ngrok", "http", PORT],
    stdout=open("/tmp/ngrok.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait for tunnel to initialize
time.sleep(5)

try:
    # Display public URL
    response = requests.get(
        "http://127.0.0.1:4040/api/tunnels",
        timeout=10
    )
    response.raise_for_status()

    for tunnel in response.json()["tunnels"]:
        print("LLM URL:", tunnel["public_url"])

    if 'llama' in globals():
        print("Llama PID:", llama.pid)
    print("Ngrok PID:", ngrok.pid)

except Exception as e:
    print(f"Failed to connect to ngrok: {e}")
    if os.path.exists("/tmp/ngrok.log"):
        print("--- Ngrok Logs ---")
        with open("/tmp/ngrok.log", "r") as f:
            print(f.read())


LLM URL: https://genre-flyer-talcum.ngrok-free.dev
Llama PID: 987
Ngrok PID: 998


In [12]:
import os
print(os.listdir('/content/models'))

['Huihui-Qwen3.5-9B-abliterated.Q4_K_M.gguf', 'Huihui-Qwen3.8-27B-abliterated-Q6_K.gguf', '.cache']
